# Private document jobs from Synapse
Install the dependency-free worker wheel as an approved workspace/pool package. Configure workspace-system-assigned identity execution and private SQL/ADLS paths first. Interactive notebooks otherwise default to your user identity; selecting an ADLS linked service does not change the SQL identity. This notebook submits references; it never calls ASE, Queue or OCR. Annotation contains PII; sanitization is pseudonymization, not guaranteed anonymity. See the deployment runbook before activation.

In [ ]:
SQL_SERVER = "approvedsql.database.usgovcloudapi.net"
SQL_DATABASE = "documents"
DOCUMENT_ID = "00000000-0000-0000-0000-000000000003"
VERSION_ID = "00000000-0000-0000-0000-000000000004"
INPUT_ETAG = '"0x000000000000001"'
EXTENSION = "pdf"
OPERATION = "redact"
LANGUAGE = "en"
RESULT_ROOT = "abfss://documents-results@approvedstorage.dfs.core.usgovcloudapi.net"
ADLS_DFS_HOST = "approvedstorage.dfs.core.usgovcloudapi.net"
ADLS_LINKED_SERVICE = "ApprovedDocumentStorageWorkspaceSAMI"

Upload an immutable, approved input version through the existing ADLS linked-service path before submission: `documents-raw/{document UUID}/{version UUID}.{extension}`. Use opaque IDs, not the original filename. Set its exact ETag above. Supported subsets are PDF, TXT, DOCX/DOC, value-focused XLSX and PNG/JPEG/TIFF. Formulas/unsupported advanced workbooks fail explicitly; do not strip features silently to bypass validation.

In [ ]:
from notebookutils import mssparkutils
from pyspark.sql import SparkSession

from presidio_document_worker.client import configure_storage, synapse_client
from presidio_document_worker.contract import Submission

spark = SparkSession.builder.getOrCreate()
configure_storage(spark, ADLS_DFS_HOST, ADLS_LINKED_SERVICE)
client = synapse_client(spark, SQL_SERVER, SQL_DATABASE, mssparkutils.credentials)
job_id = client.submit(
    Submission(DOCUMENT_ID, VERSION_ID, EXTENSION, INPUT_ETAG, OPERATION, LANGUAGE)
)
print({"job_id": job_id, "status": "submitted"})

In [ ]:
result = client.wait(job_id, timeout=3600, interval=15)
print({"job_id": job_id, "status": result["status"]})
# Submission acceptance is not processing completion. Failed/dead-letter jobs raise.
# succeeded_with_warnings remains visible; inspect the protected result manifest.

In [ ]:
import json

# Only SQL-selected complete artifact sets are authorized results.
manifest_path = RESULT_ROOT.rstrip("/") + "/" + result["result_manifest"]
manifest = json.loads(mssparkutils.fs.head(manifest_path, 65536))
print(
    {
        "job_id": job_id,
        "warnings": manifest["warnings"],
        "artifact_count": len(manifest["artifacts"]),
    }
)
# Retrieve every artifact through approved ADLS identity/linked-service paths.
# Do not display source text, annotation contents, identity maps or tokens here.